In [1]:
import sys
from pathlib import Path
import soundfile as sf

HOME = Path.home()
PIPELINE_DIR = Path("/Users/huyguy/Documents/Documents - Huys MacBook Pro/Job Materials/KwF/Inference Pipeline/KWF_Human_Presence/inference_pipeline")
RAW_DIR  = HOME / "kwf_test" / "raw_audio"          # contains Audio_Moth_5_031925/
SPLIT_OUT = HOME / "kwf_test" / "split_clips"       # new, separate from clips/

# config.py lives in inference_pipeline/, so that folder must be on the path
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

import config
print("TARGET_SAMPLE_RATE:", config.TARGET_SAMPLE_RATE)
print("SAMPLES_PER_CLIP:  ", config.SAMPLES_PER_CLIP)

from audio_split_rename import split_all_audio
print("audio_split_rename imported OK")

# What goes in
for folder in sorted(RAW_DIR.iterdir()):
    if folder.is_dir():
        for f in folder.iterdir():
            info = sf.info(str(f))
            expected = info.frames // config.SAMPLES_PER_CLIP
            print(f"\nFolder: {folder.name}  ->  recorder ID should be: {'_'.join(folder.name.split('_')[:-1])}")
            print(f"File: {f.name} | {info.samplerate} Hz | {info.duration/60:.2f} min")
            print(f"Expected full clips: {expected}  (leftover samples dropped: {info.frames % config.SAMPLES_PER_CLIP})")

print("\nOutput folder exists already:", SPLIT_OUT.exists())

TARGET_SAMPLE_RATE: 48000
SAMPLES_PER_CLIP:   144000
audio_split_rename imported OK

Folder: Audio_Moth_5_031925  ->  recorder ID should be: Audio_Moth_5
File: 20250319_163237.WAV | 48000 Hz | 87.38 min
Expected full clips: 1747  (leftover samples dropped: 96000)

Output folder exists already: False


In [2]:
import time

t0 = time.time()
split_all_audio(
    main_folder=str(RAW_DIR),
    output_folder=str(SPLIT_OUT),
)
print(f"\nRuntime: {time.time() - t0:.1f}s")
print("Clips written:", len(list(SPLIT_OUT.glob('*.wav'))))

Processed 20250319_163237.WAV: 1747 clips

Runtime: 4.9s
Clips written: 1747


In [3]:
import pandas as pd, numpy as np

clips = sorted(SPLIT_OUT.glob("*.wav"))
ORIG = RAW_DIR / "Audio_Moth_5_031925" / "20250319_163237.WAV"

# 1. Names: recorder ID and timestamp
parts = pd.Series([c.stem for c in clips]).str.extract(r"^(?P<recorder>.+)_(?P<date>\d{8})_(?P<time>\d{6})$")
parts["timestamp"] = pd.to_datetime(parts["date"] + parts["time"], format="%Y%m%d%H%M%S")
print("Unparsed names:", parts["recorder"].isna().sum())
print("Recorder IDs:", parts["recorder"].unique().tolist())
print("First:", clips[0].name, "(expected ..._20250319_163237)")
print("Last: ", clips[-1].name, "(expected ..._20250319_175955)")

# 2. Spacing between consecutive clips
gaps = parts["timestamp"].diff().dropna().dt.total_seconds()
print("\nGaps (s):", gaps.value_counts().to_dict())

# 3. Properties of every clip
infos = [sf.info(str(c)) for c in clips]
print("\nSamples per clip:", pd.Series([i.frames for i in infos]).value_counts().to_dict())
print("Sample rates:    ", pd.Series([i.samplerate for i in infos]).value_counts().to_dict())
print("Channels:        ", pd.Series([i.channels for i in infos]).value_counts().to_dict())

# 4. Does the audio match the original? Compare first, middle and last clips
print("\nAudio match vs original (max absolute difference):")
for k in [0, len(clips)//2, len(clips)-1]:
    orig, _ = sf.read(str(ORIG), start=k*config.SAMPLES_PER_CLIP, frames=config.SAMPLES_PER_CLIP)
    clip, _ = sf.read(str(clips[k]))
    print(f"  clip {k:4d} ({clips[k].name}): {np.abs(orig - clip).max():.2e}")

Unparsed names: 0
Recorder IDs: ['Audio_Moth_5']
First: Audio_Moth_5_20250319_163237.wav (expected ..._20250319_163237)
Last:  Audio_Moth_5_20250319_175955.wav (expected ..._20250319_175955)

Gaps (s): {3.0: 1746}

Samples per clip: {144000: 1747}
Sample rates:     {48000: 1747}
Channels:         {1: 1747}

Audio match vs original (max absolute difference):
  clip    0 (Audio_Moth_5_20250319_163237.wav): 0.00e+00
  clip  873 (Audio_Moth_5_20250319_171616.wav): 0.00e+00
  clip 1746 (Audio_Moth_5_20250319_175955.wav): 0.00e+00


## Observations
- Input: 1 file, 87.38 min, 48 kHz mono. Output: 1,747 clips (predicted 1,747), 4.9 s runtime.
- All names parse; recorder ID derived from the folder name (Audio_Moth_5_031925 -> Audio_Moth_5).
- All gaps exactly 3.0 s; all clips 144,000 samples, 48 kHz, mono.
- Audio of first/middle/last clip identical to the original (max diff 0.0).
- Final 96,000 samples (2.0 s) dropped by design (only full clips are written).